# 11_d Merge / Monitor Video Download Shards

`11_a`, `11_b`, `11_c` が作った shard manifest / progress JSON を確認し、完了後に canonical `raw_videos/{run_id}/download_manifest_v1.parquet` にまとめます。`12_full_cv_preprocess.ipynb` はこの canonical manifest を読みます。

途中経過を見たいだけなら `Monitor Shards` まで実行します。3 shard が終わった後に `Merge Shards` を実行します。

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


## Monitor Shards

In [ ]:
import json

from sport_pipeline.video.download_sources import merge_download_manifests, summarize_download_manifest

DOWNLOAD_SETTINGS = stage_settings(RUN_PROFILE, 'video_download')
VIDEO_REUSE_SETTINGS = stage_settings(RUN_PROFILE, 'video_reuse')
PRESERVE_EXISTING_CANONICAL_MANIFEST = bool(VIDEO_REUSE_SETTINGS.get('preserve_existing_canonical_manifest_on_merge', True))
SHARD_COUNT = int(DOWNLOAD_SETTINGS.get('batch_count') or 3)
if SHARD_COUNT != 3:
    raise RuntimeError(f'This merge notebook expects 3 shards, but profile batch_count={SHARD_COUNT}.')

def progress_bar(current: int, total: int, width: int = 28) -> str:
    if total <= 0:
        return '[----------------------------] 0.0%'
    clamped = max(0, min(current, total))
    filled = round(width * clamped / total)
    return f"[{'#' * filled}{'-' * (width - filled)}] {100.0 * clamped / total:5.1f}%"

download_dir = BASE_DIR / 'raw_videos' / RUN_ID
canonical_manifest = download_dir / 'download_manifest_v1.parquet'
shard_manifests = [
    download_dir / f'download_manifest_shard_{index + 1}_of_{SHARD_COUNT}_v1.parquet'
    for index in range(SHARD_COUNT)
]
shard_progress_paths = [
    BASE_DIR / 'reports/preflight' / f'video_download_{RUN_ID}_shard_{index + 1}_of_{SHARD_COUNT}_progress_v1.json'
    for index in range(SHARD_COUNT)
]

print('download_dir ->', download_dir)
print('canonical_manifest ->', canonical_manifest)
print('preserve_existing_canonical_manifest_on_merge =', PRESERVE_EXISTING_CANONICAL_MANIFEST)

missing = []
incomplete = []
aggregate_completed = 0
aggregate_total = int(DOWNLOAD_SETTINGS.get('max_files') or 0)
for index, manifest in enumerate(shard_manifests):
    summary = summarize_download_manifest(manifest)
    print(f'shard {index + 1}/{SHARD_COUNT} manifest:', manifest, summary)
    if not manifest.exists():
        missing.append(str(manifest))

for index, progress_path in enumerate(shard_progress_paths):
    if progress_path.exists():
        payload = json.loads(progress_path.read_text(encoding='utf-8'))
        completed = int(payload.get('overall_completed') or 0)
        total = int(payload.get('overall_total_planned') or 0)
        aggregate_completed = max(aggregate_completed, completed)
        if total:
            aggregate_total = total
        keys = ['status', 'batch_index', 'batch_count', 'num_workers', 'total_planned_this_call', 'processed_this_call', 'remaining_this_call', 'overall_completed', 'overall_total_planned', 'overall_progress_bar', 'status_counts', 'last_error']
        print(f'shard {index + 1}/{SHARD_COUNT} progress:', progress_path, {key: payload.get(key) for key in keys})
        if payload.get('status') != 'complete':
            incomplete.append(f'shard {index + 1}/{SHARD_COUNT}: {payload.get("processed_this_call")}/{payload.get("total_planned_this_call")} status={payload.get("status")}')
    else:
        print(f'shard {index + 1}/{SHARD_COUNT} progress missing:', progress_path)
        incomplete.append(f'shard {index + 1}/{SHARD_COUNT}: missing progress JSON')

print('aggregate_progress =', f'{aggregate_completed}/{aggregate_total}', progress_bar(aggregate_completed, aggregate_total))
if missing:
    print('missing_shard_manifests =')
    for path in missing:
        print('-', path)
else:
    print('all shard manifests exist; you can run the Merge Shards cell.')

## Merge Shards

In [ ]:
if missing:
    missing_text = '\n- '.join(missing)
    raise FileNotFoundError('Missing shard manifests. Finish 11_a/11_b/11_c first:\n- ' + missing_text)
if incomplete:
    incomplete_text = '\n- '.join(incomplete)
    raise RuntimeError('Shard progress is not complete yet. Re-run/finish 11_a/11_b/11_c first:\n- ' + incomplete_text)

merge_inputs = []
if PRESERVE_EXISTING_CANONICAL_MANIFEST and canonical_manifest.exists():
    merge_inputs.append(canonical_manifest)
merge_inputs.extend(shard_manifests)
summary = merge_download_manifests(merge_inputs, canonical_manifest)
print('merge_inputs =', [str(path) for path in merge_inputs])
print('merged_canonical_summary =', summary)

if bool(DOWNLOAD_SETTINGS.get('execute_default', True)):
    min_downloaded = threshold(RUN_PROFILE, 'min_downloaded_videos', 0)
    downloaded = int(summary.get('downloaded', 0))
    if downloaded < min_downloaded:
        raise RuntimeError(f'downloaded video rows too small for real run: {downloaded} < {min_downloaded}. Check shard manifests above.')

print('NEXT: notebooks/12_full_cv_preprocess.ipynb')
